# 2 · Building and running a graph

Each node in the graph binds its inputs: `From` an output, a typed-in value, or nothing (the default applies). `CompiledGraph.from_graph` checks the graph and never raises for a fault in it. `compiled.stream()` runs it as a stream of events; in a notebook, `await compiled.run()` returns the state the run ended on.

In [ ]:
from typing import Annotated

from conductor import CompiledGraph, From, Graph, GraphNode, NodeRegistry, Param, Ref, Result, Series, Static
from conductor.widgets import TextWidget

registry = NodeRegistry(category="text")


@registry.node("echo", title="Echo", description="Returns its input")
def echo(text: Annotated[str, Param(title="Input", widget=TextWidget())]) -> Annotated[str, Result(title="Output")]:
    return text


@registry.node("upper", title="Uppercase", description="Uppercases a text")
def upper(text: Annotated[str, Param(title="Input", widget=TextWidget())]) -> Annotated[str, Result(title="Output")]:
    return text.upper()


@registry.node("pair", title="Pair", description="Joins two texts")
def pair(
    a: Annotated[str, Param(title="A", widget=TextWidget())],
    b: Annotated[str, Param(title="B", widget=TextWidget())],
    separator: Annotated[str, Param(title="Separator", widget=TextWidget())] = " + ",
) -> Annotated[str, Result(title="Result")]:
    return f"{a}{separator}{b}"


@registry.node("join-all", title="Join all", description="Joins every text of a series")
def join_all(texts: Annotated[Series[str], Param(title="Texts")]) -> Annotated[str, Result(title="Result")]:
    return " ".join(texts)

## Build the graph

```text
  hello ──> shout ──┐
                    ├──> pair
  world ────────────┘
```

In [ ]:
graph = Graph(nodes=[
    echo.graph_node("hello", text="hello"),
    echo.graph_node("world", text="world"),
    upper.graph_node("shout", text=From("hello.result")),
    pair.graph_node("pair", a=From("shout.result"), b=From("world.result"), separator=" & "),
])

compiled = CompiledGraph.from_graph(graph, registry)
print("runnable:", compiled.is_runnable)
print("order:   ", compiled.execution_order)

## Ask the compiled graph

`compiled.node(node_id)` for a node, `compiled.field(ref)` for a field, and `compiled.run_plan` for where each node runs and what each input receives.

In [ ]:
print("pair.a comes from:", compiled.node("pair").graph_node.bindings["a"])
print("pair.a receives:  ", compiled.run_plan.nodes["pair"].inputs["a"].receive_mode)
print("pair.a is:        ", compiled.field(Ref("pair", "a")).dtype.describe())

## Problems, not exceptions

A broken graph compiles. Each problem has a stable `type`, and running a graph with a fatal one raises `InvalidCallError`.

In [ ]:
broken = CompiledGraph.from_graph(
    Graph(nodes=[upper.graph_node("shout", text=From("ghost.result"))]),
    registry,
)
print("runnable:", broken.is_runnable)
for problem in broken.problems:
    print(f"  {problem.type} on {problem.loc[0]}: {problem.msg}")

## Run it

A run is its `RunState`. `compiled.results(state)` is `{node_id: {output: value}}`.

In [ ]:
state = await compiled.run()
results = compiled.results(state)
for node_id, outputs in results.items():
    print(f"  {node_id}: {outputs['result']!r}")

## Fill its inputs, read its outputs

`with_inputs` fills inputs by bare name, or by address when two share a name. `compiled.outputs(state)` reads what the graph returns, by address.

In [ ]:
print("takes:  ", [str(i.name) for i in compiled.interface.inputs])
print("returns:", [str(o.name) for o in compiled.interface.outputs])

ready = compiled.with_inputs(**{"hello.text": "goodbye"})
state = await ready.run()
ready.outputs(state)

## Watch the events

In [ ]:
async with compiled.stream() as events:
    async for event in events:
        match event.type:
            case "node_start":
                print(f"  start     {event.node_id}")
            case "node_complete":
                print(f"  complete  {event.node_id}: {event.outputs['result']!r}")
            case "run_end":
                print(f"  done      {sorted(compiled.results(event.state))}")
            case other:
                print(f"  {other}")

## Once per row

A list on an input declared for one value runs the node once per value. A `Series[X]` input takes the whole series in one call. A list isn't what `text: str` says, so this input is written as a plain `GraphNode`.

In [ ]:
rows = CompiledGraph.from_graph(
    Graph(nodes=[
        GraphNode(id="shout", type="upper", version=1, bindings={"text": Static(["one", "two", "three"])}),
        join_all.graph_node("all", texts=From("shout.result")),
    ]),
    registry,
)
print(rows.run_plan)

async with rows.stream() as events:
    async for event in events:
        if event.type == "node_progress":
            print(f"  {event.node_id}: {event.done} of {event.total}")
        elif event.type == "run_end":
            results = rows.results(event.state)
            print("  shout:", results["shout"]["result"]["values"])
            print("  all:  ", results["all"]["result"])

## Save it

A `Graph` reads back the JSON it wrote.

In [ ]:
text = graph.model_dump_json()
print(text)
assert Graph.model_validate_json(text) == graph